In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import create_engine
import requests
import scipy
from scipy import stats
from scipy import optimize
from scipy import signal
from IPython.display import display
import csv

In [5]:
#Cleaning Nav_History parse dates to datetime, sort by amfi_code + date,
#forward-fill missing NAV for holidays/weekends, remove duplicates, validate NAV > 0.

#parsing dates to datetime
nav_history = pd.read_csv("02_nav_history.csv",parse_dates=['date'])

#Sorting by amfi_code + date
nav_history = nav_history.sort_values(['amfi_code','date']).reset_index(drop=True)

#Removing Duplicates 
nav_history = nav_history.drop_duplicates(subset=['amfi_code','date'],keep='first')

#Forward-fill for holidays/weekends
all_dates = pd.date_range(start = nav_history['date'].min(),end = nav_history['date'].max(),freq = 'D')

filled_chunks=[]
for code,group in nav_history.groupby('amfi_code'):
    group = group.set_index('date')
    group = group.reindex(all_dates)
    group['amfi-code'] = code
    group['nav'] = group['nav'].ffill()
    group.index.name = 'date'
    group = group.reset_index()
    filled_chunks.append(group)

nav_history = pd.concat(filled_chunks,ignore_index=True)

#Removing Duplicates
print("Duplicates_Before:",nav_history.duplicated(subset = ['amfi_code','date']).sum())

nav_history = nav_history.drop_duplicates(subset = ['amfi_code','date'], keep = 'first')
print("Duplicates_After:",nav_history.duplicated(subset = ['amfi_code','date']).sum())


#Validating Nav > 0
invalid_nav = nav_history[nav_history['nav']<=0]
print(f"Invalid NAV rows (<=0): {len(invalid_nav)}")
print(invalid_nav)

#Final check & summarry 
print("Shape:",nav_history.shape)
print("Data Range:", nav_history['date'],"-->",nav_history['date'].max())
print("AMFI_Codes:", nav_history['amfi_code'].nunique())
print("Missing Nav:",nav_history.duplicated(subset=['amfi_code','date']).sum())
print("\nSample")
nav_history.head(10)

Duplicates_Before: 17862
Duplicates_After: 0
Invalid NAV rows (<=0): 0
Empty DataFrame
Columns: [date, amfi_code, nav, amfi-code]
Index: []
Shape: (46458, 4)
Data Range: 0       2022-01-03
1       2022-01-04
2       2022-01-05
3       2022-01-06
4       2022-01-07
           ...    
64315   2026-05-25
64316   2026-05-26
64317   2026-05-27
64318   2026-05-28
64319   2026-05-29
Name: date, Length: 46458, dtype: datetime64[ns] --> 2026-05-29 00:00:00
AMFI_Codes: 40
Missing Nav: 0

Sample


,date,amfi_code,nav,amfi-code
0,2022-01-03,100016.0,520.4608,100016
1,2022-01-04,100016.0,515.0971,100016
2,2022-01-05,100016.0,521.7239,100016
3,2022-01-06,100016.0,515.7880,100016
4,2022-01-07,100016.0,515.1639,100016
5,2022-01-08,NaN,515.1639,100016
6,2022-01-09,NaN,515.1639,100016
7,2022-01-10,100016.0,510.7136,100016
8,2022-01-11,100016.0,513.5542,100016
9,2022-01-12,100016.0,512.3195,100016


In [6]:
#Cleaning investor_transactions.csv — standardise transaction_type values (SIP/Lumpsum/Redemption), 
#validate amount > 0, fix date formats, check KYC status enum values.
#loading data
investor_transactions = pd.read_csv("08_investor_transactions.csv")

#Standardising transaction_type values (SIP/Lumpsum/Redemption)
investor_transactions = pd.read_csv("08_investor_transactions.csv")
transactions_mapping = {
    #SIP variants 
    'sip'        : 'SIP',
    'SIP'        : 'SIP',
    'Sip'  :'SIP',
    's.i.p' :'SIP',
    'systematic':'SIP',
    'systematic investment plan' : 'SIP',

    #Lumpsum Variants 
    'lumpsum':'Lumpsum',
    'LUMPSUM':'Lumpsum',
    'lump sum':'Lumpsum',
    'lump_sum':'Lumpsum',
    'one time':'Lumpsum',
    'onetime':'Lumpsum',
    'one-time':'Lumpsum',

    #Redemption Var''
    'redemption':'Redemption',
    'REDEMPTION':'Redemption',
    'redeem':'Redemption',
    'Redeem':'Redemption',
    'withdrawal':'Redemption',
    'withdraw':'Redemption',
    
}

investor_transactions['transaction_type'] = investor_transactions['transaction_type'].str.strip()
investor_transactions['transaction_type'] = investor_transactions['transaction_type'].map(transactions_mapping)
invalid_types = investor_transactions[investor_transactions['transaction_type'].isnull()]
if not invalid_types.empty:
    print(f"{len(invalid_types)} unkown transaction_type values:")
    print(invalid_types)

investor_transactions = investor_transactions[investor_transactions['transaction_type'].notna()].reset_index(drop = True)

#validating amount > 0
investor_transactions['amount_inr'] = pd.to_numeric(investor_transactions['amount_inr'], errors = 'coerce')
investor_transactions = investor_transactions[investor_transactions['amount_inr']>0].reset_index(drop=True)

#Fixing Date Formats
def parse_date_flexible(s):
    for fmt in ['%Y-%m-%d','%d-%m-%Y','%d/%m/%Y','%m/%d%Y','%d-%b-%Y','%Y/%m/%d']:
        try:
            return pd.to_datetime(s,format=fmt)
        except:
            continue
        return pd.to_datetime(s,dayfirst=True,errors='coerce')

investor_transactions['transaction_date'] = parse_date_flexible(investor_transactions['transaction_date'])
investor_transactions = investor_transactions[investor_transactions['transaction_date'].notna()].reset_index(drop =True)
#checking KYC status enum values
valid_kyc = {'VERIFIED','PENDING','REJECTED','EXPIRED'}
investor_transactions['kyc_status'] = investor_transactions['kyc_status'].str.upper()
investor_transactions = investor_transactions[investor_transactions['kyc_status'].isin(valid_kyc)].reset_index(drop = True)

print(f"Rows: {investor_transactions.shape[0]:,}")
print(f" Transaction types: {investor_transactions['transaction_type'].value_counts().to_dict()}")
print(f" KYC status: {investor_transactions['kyc_status'].value_counts().to_dict()}")
print(f"Amount Range: ₹{investor_transactions['amount_inr'].min():,.2f} --> ₹{investor_transactions['amount_inr'].max():,.2f}")
print(f"Date Range: {investor_transactions['transaction_date'].min().date()} --> {investor_transactions['transaction_date'].max().date}")
print(f" Nulls: {investor_transactions.isnull().sum().to_dict()}")
investor_transactions.head(10)

13062 unkown transaction_type values:
      investor_id transaction_date  amfi_code transaction_type  amount_inr  \
1       INV002952       2024-01-01     148567              NaN      392882   
4       INV004691       2024-01-01     119094              NaN        8682   
10      INV002592       2024-01-01     101208              NaN      411860   
14      INV000368       2024-01-01     118636              NaN      339882   
21      INV003329       2024-01-01     101207              NaN      295055   
...           ...              ...        ...              ...         ...   
32767   INV003199       2025-05-30     120843              NaN      151988   
32768   INV004993       2025-05-30     149322              NaN       84982   
32770   INV000734       2025-05-30     149322              NaN      127218   
32773   INV003340       2025-05-30     101207              NaN      168029   
32777   INV001852       2025-05-30     118633              NaN      226721   

                state    

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,VERIFIED
1,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,VERIFIED
2,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,PENDING
3,INV001497,2024-01-01,101208,SIP,3295,Maharashtra,Mumbai,T30,36-45,Male,56.8,Mandate,VERIFIED
4,INV000786,2024-01-01,101208,SIP,15047,Madhya Pradesh,Bhopal,B30,26-35,Male,17.9,Mandate,VERIFIED
5,INV000616,2024-01-01,100016,SIP,933,Maharashtra,Nashik,B30,36-45,Male,50.4,Mandate,VERIFIED
6,INV003670,2024-01-01,119120,SIP,10672,Punjab,Chandigarh,T30,36-45,Male,38.4,Net Banking,VERIFIED
7,INV002054,2024-01-01,148567,SIP,1111,Telangana,Hyderabad,T30,26-35,Male,17.8,Net Banking,VERIFIED
8,INV001023,2024-01-01,118636,SIP,4865,Gujarat,Ahmedabad,T30,36-45,Male,27.1,Net Banking,VERIFIED
9,INV002893,2024-01-01,120843,SIP,8472,Telangana,Hyderabad,T30,56+,Male,69.4,UPI,PENDING


In [116]:
#Clean scheme_performance.csv — validate all return values are numeric, flag anomalies, check expense_ratio range (0.1% – 2.5%).
scheme_performance = pd.read_csv("07_scheme_performance.csv")
#validate all return values are numeric
print(f"Return_1yr_pct: {scheme_performance['return_1yr_pct'].unique()}")
print(f"Return_3yr_pct: {scheme_performance['return_3yr_pct'].unique()}")
print(f"Return_5yr_pct: {scheme_performance['return_5yr_pct'].unique()}")

scheme_performance['return_1yr_pct_clean'] = pd.to_numeric(scheme_performance['return_1yr_pct'],errors= 'coerce')
non_numeric_mask1 = scheme_performance['return_1yr_pct_clean'].isnull()&scheme_performance['return_1yr_pct'].notna()
non_numeric_rows1 = scheme_performance[non_numeric_mask1]

print(f"return_1yr_pct Non-numeric rows: {len(non_numeric_rows1)}")


scheme_performance['return_3yr_pct_clean'] = pd.to_numeric(scheme_performance['return_3yr_pct'],errors= 'coerce')
non_numeric_mask2 = scheme_performance['return_3yr_pct_clean'].isnull()&scheme_performance['return_3yr_pct'].notna()
non_numeric_rows2 = scheme_performance[non_numeric_mask2]

print(f"return_3yr_pct Non-numeric rows: {len(non_numeric_rows2)}")


scheme_performance['return_5yr_pct_clean'] = pd.to_numeric(scheme_performance['return_5yr_pct'],errors= 'coerce')
non_numeric_mask3 = scheme_performance['return_5yr_pct_clean'].isnull()&scheme_performance['return_5yr_pct'].notna()
non_numeric_rows3 = scheme_performance[non_numeric_mask3]

print(f"return_5yr_pct Non-numeric rows: {len(non_numeric_rows3)}")


# check expense_ratio range (0.1% – 2.5%)
scheme_performance['expense_ratio_pct'] = pd.to_numeric(scheme_performance['expense_ratio_pct'],errors= 'coerce')
scheme_performance = scheme_performance.dropna(subset= ['expense_ratio_pct']).reset_index(drop = True)
Ratio_Min = 0.1
Ratio_Max = 2.5

for expense_ratio_pct in scheme_performance:
    scheme_performance[f'{expense_ratio_pct}_boundary_flag'] = ~scheme_performance['expense_ratio_pct'].between(Ratio_Min,Ratio_Max)

for expense_ratio_pct in expense_ratio_pct:
    scheme_performance[f'{expense_ratio_pct}_zscore'] = np.abs(stats.zscore(scheme_performance['expense_ratio_pct'],nan_policy='omit'))
    scheme_performance['expense_ratio_pct_zscore_flag'] = scheme_performance[f'{expense_ratio_pct}_zscore']>3

print(scheme_performance['expense_ratio_pct'].head())
print(scheme_performance['expense_ratio_pct_boundary_flag'].head())
print(scheme_performance['expense_ratio_pct_zscore_flag'].head())



for expense_ratio_pct in expense_ratio_pct:
    Q1,Q3 = scheme_performance['expense_ratio_pct'].quantile(0.25),scheme_performance['expense_ratio_pct'].quantile(0.75)
    IQR = Q3-Q1
    scheme_performance['expense_ratio_pct_iqr_flag'] = ~scheme_performance['expense_ratio_pct'].between(Q1-1.5*IQR,Q3+1.5*IQR)
print(scheme_performance['expense_ratio_pct_iqr_flag'].head())


all_flags = [c for c in scheme_performance.columns if c.endswith('_flag')]
scheme_performance['is_anomaly'] = scheme_performance[all_flags].any(axis=1)

#Summary
total_rows = len(scheme_performance)
clean_rows = (~scheme_performance['is_anomaly']).sum()
anomaly_rows = scheme_performance['is_anomaly'].sum()
anomaly_rate = anomaly_rows/total_rows*100

               
print(f"Total Rows: {total_rows:,}")
print(f"Clean Rows: {clean_rows:,}")
print(f"Anomaly Rows: {anomaly_rows:,}")
print(f"Anomaly Rate: {anomaly_rate:.2f}%")




Return_1yr_pct: [12.42 15.25 24.56 20.59  5.34 10.94 11.48 15.43 19.98  6.83 15.63 14.12
 14.02 16.67  8.89 15.84 14.42 21.3  10.14  6.19 14.37 17.12 15.74  4.26
 11.82 13.95 14.88 21.97 14.82 24.93  6.18 13.76 16.3  17.43 15.12 14.91
 11.16 11.96 20.2 ]
Return_3yr_pct: [12.36 11.3  23.39 23.14  6.07 14.84 13.38 16.58 15.29  7.37 11.54 14.41
 18.08 14.76  7.68 14.   12.33 20.15 11.77  5.31 12.25 18.23 15.65  6.18
 11.84 12.14 15.18 20.98 13.78 22.38  5.14 12.1  15.61 15.34 14.81 14.56
 13.58 12.82 17.16 20.08]
Return_5yr_pct: [14.45 14.23 20.67 21.82  5.43 11.32 13.48 17.69 15.85  6.41 11.46 13.02
 17.55 12.6   7.94 14.7  14.72 21.88 12.31  8.71 13.42 17.75 13.5   8.26
 14.14 13.66 18.94 22.62 12.86 23.8   7.95 11.31 15.86 15.78 12.68 15.68
 14.26 12.35 19.   20.61]
return_1yr_pct Non-numeric rows: 0
return_3yr_pct Non-numeric rows: 0
return_5yr_pct Non-numeric rows: 0
0    1.54
1    0.66
2    1.43
3    0.72
4    0.77
Name: expense_ratio_pct, dtype: float64
0    False
1    False
2    F